# Time Series Analysis — CA Mensuel SAGAP
## Auto-ARIMA | Décomposition | Diagnostic | Prévision 2026

In [ ]:
!pip install pmdarima statsmodels matplotlib pandas numpy scikit-learn clickhouse-connect --quiet
print('✅ Installation OK')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import clickhouse_connect
import warnings

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pmdarima as pm

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Imports OK')

ModuleNotFoundError: No module named 'pmdarima'

## 1. Chargement depuis ClickHouse

In [ ]:
client = clickhouse_connect.get_client(
    host='clickhouse', port=8123,
    user='default', password='clickhouse123',
    database='gold'
)

df = client.query_df("""
SELECT d.Year AS year, d.Month AS month,
       SUM(f.LinePrice) AS total_sales
FROM gold.fact_sales AS f
JOIN gold.dim_date AS d ON f.DateSK = d.DateSK
WHERE f.DocumentTypeSK IN (4, 1, 5)
  AND f.LinePrice IS NOT NULL
GROUP BY year, month
ORDER BY year, month
""")

df['Date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

# Garder Jan 2022 → Déc 2025
df = df[
    (df['Date'] >= '2022-01-01') & (df['Date'] < '2026-01-01')
].reset_index(drop=True)

df.set_index('Date', inplace=True)
df.index.freq = 'MS'
df = df[['total_sales']].rename(columns={'total_sales': 'Value'})

print(f'✅ {len(df)} mois chargés')
print(f'   Période : {df.index[0].strftime("%b %Y")} → {df.index[-1].strftime("%b %Y")}')
print(f'   CA moyen : {df.Value.mean():,.0f} DT')
df.head()

## 2. Time Series Plot

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.Value, color='steelblue', marker='o', linewidth=2)
plt.title('CA Mensuel SAGAP — Jan 2022 → Déc 2025')
plt.xlabel('Date')
plt.ylabel('CA (DT)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Min  : {df.Value.min():,.0f} DT — {df.Value.idxmin().strftime("%b %Y")}')
print(f'Max  : {df.Value.max():,.0f} DT — {df.Value.idxmax().strftime("%b %Y")}')
print(f'Std  : {df.Value.std():,.0f} DT')

## 3. Interpolation des valeurs manquantes

In [ ]:
print(f'Valeurs manquantes avant : {df.Value.isna().sum()}')
df.Value = df.Value.interpolate(method='linear')
print(f'Valeurs manquantes après : {df.Value.isna().sum()}')
print('✅ Interpolation OK')

## 4. Détection et nettoyage des outliers (Z-score > 3)

In [ ]:
z = np.abs((df.Value - df.Value.mean()) / df.Value.std())
n_outliers = (z > 3).sum()
print(f'Outliers détectés (z > 3) : {n_outliers}')

df['Value_clean'] = np.where(z > 3, df.Value.median(), df.Value)

if n_outliers > 0:
    print('Mois remplacés par la médiane :')
    print(df[z > 3][['Value', 'Value_clean']].to_string())
else:
    print('✅ Aucun outlier — données propres')

# Visualiser
plt.figure(figsize=(12, 4))
plt.plot(df.Value,       color='steelblue', alpha=0.5, label='Original')
plt.plot(df.Value_clean, color='orange',    linewidth=2, label='Nettoyée')
plt.title('Série originale vs Nettoyée')
plt.ylabel('CA (DT)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Test de Stationnarité — ADF (Augmented Dickey-Fuller)

In [ ]:
adf = adfuller(df.Value_clean)
print('=== Test ADF ===')
print(f'ADF Statistic : {adf[0]:.4f}')
print(f'p-value       : {adf[1]:.4f}')
print(f'Valeurs critiques :')
for k, v in adf[4].items():
    print(f'  {k} : {v:.4f}')

if adf[1] < 0.05:
    print('\n✅ Série STATIONNAIRE → d=0 pour ARIMA')
else:
    print('\n⚠️  Série NON STATIONNAIRE → différenciation nécessaire (d=1)')

## 6. Différenciation (si nécessaire)

In [ ]:
df['Value_diff'] = df.Value_clean.diff().dropna()

# Vérifier stationnarité après différenciation
adf_diff = adfuller(df.Value_diff.dropna())
print(f'ADF après diff : p-value = {adf_diff[1]:.4f}')
print('✅ Stationnaire après diff' if adf_diff[1] < 0.05 else '⚠️  Toujours non stationnaire')

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(df.Value_clean, color='steelblue', marker='o')
axes[0].set_title('Série originale')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].plot(df.Value_diff.dropna(), color='orange', marker='o')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Série différenciée (d=1)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

## 7. Décomposition Saisonnière

In [ ]:
decomp = seasonal_decompose(df.Value_clean, model='additive', period=12)
fig = decomp.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Décomposition Saisonnière — CA Mensuel SAGAP', fontsize=13, y=1.02)

# Formater les axes Y
for ax in fig.axes:
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

# Analyse de la saisonnalité
print('\nSaisonnalité mensuelle (écart vs moyenne) :')
seasonal_vals = decomp.seasonal[:12]
mois = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
for m, v in zip(mois, seasonal_vals.values):
    sign = '+' if v >= 0 else ''
    print(f'  {m} : {sign}{v:,.0f} DT')

## 8. ACF / PACF — Identifier les paramètres ARIMA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df.Value_clean,  lags=20, ax=axes[0], title='ACF  — Autocorrélation')
plot_pacf(df.Value_clean, lags=20, ax=axes[1], title='PACF — Autocorrélation Partielle')
plt.tight_layout()
plt.show()

print('Lecture :')
print('  ACF  → pic au lag 12 = saisonnalité annuelle ✅')
print('  PACF → lags significatifs → ordre AR (p)')

## 9. Split Train / Test (12 derniers mois en test)

In [ ]:
train = df.Value_clean[:-12]
test  = df.Value_clean[-12:]

print(f'Train : {len(train)} mois ({train.index[0].strftime("%b %Y")} → {train.index[-1].strftime("%b %Y")})')
print(f'Test  : {len(test)} mois ({test.index[0].strftime("%b %Y")} → {test.index[-1].strftime("%b %Y")})')

plt.figure(figsize=(12, 5))
plt.plot(train, label='Train', color='steelblue', marker='o')
plt.plot(test,  label='Test',  color='orange',    marker='o')
plt.title('Split Train / Test')
plt.ylabel('CA (DT)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Auto-ARIMA — Sélection automatique des meilleurs paramètres

In [ ]:
print('🔍 Recherche automatique des meilleurs paramètres ARIMA...')
print('   (peut prendre 1-2 minutes)\n')

model = pm.auto_arima(
    train,
    seasonal=True,
    m=12,                    # saisonnalité mensuelle
    d=None,                  # auto-détection de la différenciation
    D=None,                  # auto-détection saisonnière
    max_p=3, max_q=3,
    max_P=2, max_Q=2,
    information_criterion='aic',
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True            # plus rapide
)

print(f'\n✅ Meilleur modèle : ARIMA{model.order} × {model.seasonal_order}')
print(f'   AIC : {model.aic():.2f}')
model.summary()

## 11. Prédiction sur la période Test

In [ ]:
# Prédiction avec intervalles de confiance
forecast_vals, conf_int = model.predict(n_periods=12, return_conf_int=True)
forecast = pd.Series(forecast_vals, index=test.index)
ci_low   = pd.Series(conf_int[:, 0], index=test.index)
ci_high  = pd.Series(conf_int[:, 1], index=test.index)

print('Prédictions vs Réel :')
print(f'{"Mois":<12} {"Réel":>14} {"Prédit":>14} {"Écart%":>8}')
print('-'*52)
for date in test.index:
    reel  = test[date]
    pred  = forecast[date]
    ecart = (pred - reel) / reel * 100
    print(f'{date.strftime("%b %Y"):<12} {reel:>14,.0f} {pred:>14,.0f} {ecart:>7.1f}%')

## 12. Visualisation Forecast vs Actual

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train,    label='Train',  color='steelblue', linewidth=1.5)
plt.plot(test,     label='Réel',   color='black',     marker='o', linewidth=2)
plt.plot(forecast, label='Prédit', color='orange',    marker='s', linestyle='--', linewidth=2)

# Intervalle de confiance 95%
plt.fill_between(test.index, ci_low, ci_high,
                 alpha=0.2, color='orange', label='IC 95%')

plt.title('Forecast vs Actual — Auto-ARIMA')
plt.xlabel('Date')
plt.ylabel('CA (DT)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Métriques d'évaluation (sans R²)

In [ ]:
mae   = mean_absolute_error(test, forecast)
mse   = mean_squared_error(test, forecast)
rmse  = np.sqrt(mse)
mape  = np.mean(np.abs((test - forecast) / test)) * 100
smape = np.mean(2 * np.abs(test - forecast) / (np.abs(test) + np.abs(forecast))) * 100
bias  = np.mean(forecast.values - test.values)

print('='*50)
print(f'📊 MÉTRIQUES Auto-ARIMA{model.order} × {model.seasonal_order}')
print('='*50)
print(f'MAE   : {mae:>15,.0f} DT  ← erreur absolue moyenne')
print(f'RMSE  : {rmse:>15,.0f} DT  ← pénalise les grosses erreurs')
print(f'MAPE  : {mape:>15.2f} %   ← erreur % moyenne')
print(f'SMAPE : {smape:>15.2f} %   ← MAPE symétrique')
print(f'Biais : {bias:>15,.0f} DT  ← {"surestimation" if bias > 0 else "sous-estimation"}')
print('='*50)

if mape < 5:
    print('\n✅ Excellent (MAPE < 5%)')
elif mape < 10:
    print('\n✅ Bon (5% < MAPE < 10%)')
elif mape < 20:
    print('\n⚠️  Acceptable (10% < MAPE < 20%)')
else:
    print('\n❌ Mauvais (MAPE > 20%)')

## 14. Diagnostic des résidus

In [ ]:
residuals = test.values - forecast.values

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Résidus dans le temps
axes[0,0].plot(test.index, residuals, color='steelblue', marker='o')
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Résidus dans le temps')
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Distribution des résidus
axes[0,1].hist(residuals, bins=8, color='steelblue', edgecolor='white')
axes[0,1].axvline(0, color='red', linestyle='--')
axes[0,1].set_title('Distribution des résidus')
axes[0,1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Réel vs Prédit
axes[1,0].scatter(test.values, forecast.values, color='steelblue', alpha=0.7)
min_val = min(test.min(), forecast.min())
max_val = max(test.max(), forecast.max())
axes[1,0].plot([min_val, max_val], [min_val, max_val], 'r--', label='Ligne parfaite')
axes[1,0].set_title('Réel vs Prédit')
axes[1,0].set_xlabel('Réel')
axes[1,0].set_ylabel('Prédit')
axes[1,0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[1,0].legend()

# Erreurs % par mois
mape_par_mois = np.abs((test.values - forecast.values) / test.values) * 100
mois_labels   = [d.strftime('%b %Y') for d in test.index]
colors = ['green' if m < 10 else 'orange' if m < 20 else 'red' for m in mape_par_mois]
axes[1,1].bar(mois_labels, mape_par_mois, color=colors)
axes[1,1].axhline(10, color='orange', linestyle='--', label='Seuil 10%')
axes[1,1].axhline(5,  color='green',  linestyle='--', label='Seuil 5%')
axes[1,1].set_title('MAPE par mois (%)')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].legend(fontsize=8)

plt.suptitle('Diagnostic des Résidus — Auto-ARIMA', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Résidu moyen  : {residuals.mean():,.0f} DT')
print(f'Résidu std    : {residuals.std():,.0f} DT')
print(f'Résidu max abs: {np.abs(residuals).max():,.0f} DT')

## 15. Prévision 2026 — 12 prochains mois

In [ ]:
print('Réentraînement sur toute la série (Jan 2022 → Déc 2025)...')

model_full = pm.auto_arima(
    df.Value_clean,
    seasonal=True, m=12,
    trace=False,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print(f'✅ Modèle final : ARIMA{model_full.order} × {model_full.seasonal_order}')

# Prédiction Jan 2026 → Déc 2026
future_vals, future_ci = model_full.predict(n_periods=12, return_conf_int=True)
future_index = pd.date_range(
    df.index[-1] + pd.DateOffset(months=1), periods=12, freq='MS'
)

future_series  = pd.Series(future_vals,        index=future_index)
future_ci_low  = pd.Series(future_ci[:, 0],    index=future_index)
future_ci_high = pd.Series(future_ci[:, 1],    index=future_index)

print(f'\nPrévisions Jan 2026 → Déc 2026 :')
for date, val in future_series.items():
    print(f'  {date.strftime("%b %Y")} → {val:>15,.2f} DT')

ca_2025 = df.Value_clean[df.index.year == 2025].sum()
ca_2026 = future_series.sum()
print(f'\nCA réel  2025 : {ca_2025:>15,.2f} DT')
print(f'CA prévu 2026 : {ca_2026:>15,.2f} DT')
print(f'Croissance    : {(ca_2026 - ca_2025)/ca_2025*100:>15.2f} %')

## 16. Full Forecast — Historique + Prévision 2026

In [ ]:
plt.figure(figsize=(14, 6))

# Historique
plt.plot(df.Value_clean, label='CA Réel (historique)',
         color='steelblue', marker='o', linewidth=2)

# Prévision 2026
plt.plot(future_series, label=f'CA Prédit 2026 (Auto-ARIMA{model_full.order})',
         color='orange', marker='s', linestyle='--', linewidth=2)

# Intervalle de confiance
plt.fill_between(future_index, future_ci_low, future_ci_high,
                 alpha=0.2, color='orange', label='IC 95%')

# Ligne de coupure
plt.axvline(df.index[-1], color='red', linestyle=':', linewidth=1.5,
            label='Déc 2025 — Fin données réelles')

plt.title(f'Full Forecast — CA Réel + Prévision 2026 | Auto-ARIMA{model_full.order}')
plt.xlabel('Date')
plt.ylabel('CA (DT)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 17. CA Annuel — 2024 Réel | 2025 Réel | 2026 Prédit

In [ ]:
ca_2024 = df.Value_clean[df.index.year == 2024].sum()
ca_2025 = df.Value_clean[df.index.year == 2025].sum()
ca_2026 = future_series.sum()
croissance = (ca_2026 - ca_2025) / ca_2025 * 100

print('='*55)
print('📋 PRÉVISION CA ANNUEL 2026')
print('='*55)
print(f'CA réel 2024      : {ca_2024:>15,.2f} DT')
print(f'CA réel 2025      : {ca_2025:>15,.2f} DT')
print(f'CA prévu 2026     : {ca_2026:>15,.2f} DT')
print(f'Croissance 25→26  : {croissance:>15.2f} %')
print(f'Modèle            : Auto-ARIMA{model_full.order}')
print(f'MAPE test         : {mape:.2f}%')
print('='*55)

# Graphe annuel
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['2024\n(réel)', '2025\n(réel)', '2026\n(prédit)'],
              [ca_2024, ca_2025, ca_2026],
              color=['steelblue', 'steelblue', 'orange'])
for bar, val in zip(bars, [ca_2024, ca_2025, ca_2026]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(ca_2024, ca_2025, ca_2026)*0.01,
            f'{val:,.0f} DT', ha='center', fontsize=9)
ax.set_title('CA Annuel : 2024 Réel | 2025 Réel | 2026 Prévu')
ax.set_ylabel('CA (DT)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 18. Écriture dans ClickHouse Gold

In [ ]:
df_pred = pd.DataFrame({
    'date':            future_index,
    'year':            future_index.year,
    'month':           future_index.month,
    'predicted_sales': future_vals,
    'is_prediction':   1
})

client2 = clickhouse_connect.get_client(
    host='clickhouse', port=8123,
    user='default', password='clickhouse123', database='gold'
)

client2.command("""
    CREATE TABLE IF NOT EXISTS gold.ml_predictions (
        date Date, year Int32, month Int32,
        predicted_sales Float64, is_prediction Int8
    ) ENGINE = MergeTree() ORDER BY date
""")

client2.command('TRUNCATE TABLE gold.ml_predictions')
client2.insert_df('gold.ml_predictions',
    df_pred[['date', 'year', 'month', 'predicted_sales', 'is_prediction']])

check = client2.query_df('SELECT * FROM gold.ml_predictions ORDER BY date')
print(f'✅ {len(check)} prédictions écrites dans ClickHouse Gold')
print('   Table : gold.ml_predictions\n')
print(check.to_string(index=False))